# 04 — Model Comparison: Linear Regression vs Decision Tree vs Random Forest

**Goal (game plan Week 5):** try Decision Tree and Random Forest regressors, compare them with the Linear Regression baseline, and document strengths and weaknesses.
**Author:** Ethan Chung — IDX Exchange Data Science, Fall 2026

All three models go through the identical procedure in `src/models.py` (IDX Best Practices §01, §09):
1. **Selection on the 2026-03 validation month only:** training window (3 / 6 / 12 / 24 months) and a small hyperparameter grid, keeping the lowest validation MdAPE.
2. **Test once** on 2026-04 after refitting on the months right before it.
3. **Rolling backtest** on 2026-03, 2026-02, 2026-01.

Same features, same leakage-safe preprocessing pipeline, fixed random seeds.

**Run `python src/models.py --features week5` first** (about 17 minutes): it trains every model and saves them to `models/` and their metrics to `docs/metrics_*.csv`. This notebook only loads those results.

## 1. Setup

In [ ]:
import os
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import evaluation
import models
import preprocessing

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
NAMES = list(models.MODELS)
LABELS = {"baseline_linear_regression": "Linear Regression", "decision_tree": "Decision Tree", "random_forest": "Random Forest"}
METRICS = ["R2", "MdAPE", "MAPE", "MAE", "RMSE", "within_10pct", "median_pct_error"]

clean = preprocessing.load_clean("../data/processed/sfr_clean.parquet")
FEATURE_SET = "week5"   # the Week 5 comparison; Week 6 feature engineering is in 04b
metrics = pd.concat([pd.read_csv(f"../docs/metrics_{models.artifact_stem(n, FEATURE_SET)}.csv") for n in NAMES], ignore_index=True)

# Reload each saved model and score its own test set (test month, cutoffs from its training window)
results = {}
for n in NAMES:
    art = joblib.load(f"../models/{models.artifact_stem(n, FEATURE_SET)}.joblib")
    _, test, _ = preprocessing.make_final_split(clean, art["train_info"]["train_months"])
    results[n] = {"model": art["model"], "params": art["params"], "info": art["train_info"],
                  "test": test, "pred": models.predict_dollars(art["model"], test)}
{LABELS[n]: (r["info"]["train_months"], r["params"]) for n, r in results.items()}

## 2. What was chosen on validation

In [ ]:
val = metrics[metrics["stage"] == "validation (selection)"]
best_val = val.loc[val.groupby("model")["MdAPE"].idxmin(), ["model", "train_months", "params", "R2", "MdAPE", "MAPE"]]
best_val.assign(model=best_val["model"].map(LABELS))

In [ ]:
# Validation MdAPE by window length (best hyperparameters per window)
by_window = val.groupby(["model", "train_months"])["MdAPE"].min().unstack("model").rename(columns=LABELS)
ax = by_window.plot(marker="o", figsize=(7, 3.5), title="Validation MdAPE by training window (2026-03)")
ax.set_xlabel("training window (months)"); ax.set_ylabel("MdAPE (%)"); ax.set_xticks(models.WINDOWS)
plt.tight_layout(); plt.show()

## 3. Side-by-side test results (2026-04)

In [ ]:
test_table = metrics[metrics["stage"] == "test"].set_index("model")[["train_months"] + METRICS]
test_table.rename(index=LABELS)

### Is the improvement more than noise?

Paired bootstrap on the test homes that every model scored (IDX Best Practices §09): each round resamples homes and recomputes both MdAPEs on the same sample. If the 95% interval for the improvement excludes 0, the gain is real.

In [ ]:
common = set.intersection(*(set(r["test"]["ListingKey"]) for r in results.values()))
aligned = {n: pd.Series(r["pred"], index=r["test"]["ListingKey"].to_numpy()).loc[sorted(common)] for n, r in results.items()}
y = clean.set_index("ListingKey").loc[sorted(common), "ClosePrice"]
rows = []
for n in ["decision_tree", "random_forest"]:
    b = evaluation.paired_bootstrap_mdape(y, aligned["baseline_linear_regression"], aligned[n])
    rows.append({"model": LABELS[n], **b})
print(f"{len(common):,} test homes scored by all models")
pd.DataFrame(rows).rename(columns={"mdape_a": "baseline MdAPE", "mdape_b": "model MdAPE"})

## 4. Stability: rolling backtest

In [ ]:
hist = metrics[metrics["stage"].isin(["test", "backtest"])][["model", "test_month", "R2", "MdAPE"]]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for n in NAMES:
    h = hist[hist["model"] == n].sort_values("test_month")
    axes[0].plot(h["test_month"], h["R2"], marker="o", label=LABELS[n])
    axes[1].plot(h["test_month"], h["MdAPE"], marker="o", label=LABELS[n])
axes[0].set_title("R² by test month"); axes[1].set_title("MdAPE (%) by test month")
axes[1].set_ylim(0, None); axes[0].legend()
plt.tight_layout(); plt.show()
hist.pivot(index="test_month", columns="model", values="MdAPE").rename(columns=LABELS)

## 5. Where each model is better or worse

In [ ]:
band_tables = []
for n, r in results.items():
    t = evaluation.metrics_by_group(r["test"]["ClosePrice"], r["pred"], evaluation.price_bands(r["test"]["ClosePrice"]))
    band_tables.append(t.assign(model=LABELS[n]))
bands = pd.concat(band_tables)
bands["band"] = bands["group"].str.split(" ").str[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
bands.pivot(index="band", columns="model", values="MdAPE").plot(kind="bar", ax=axes[0], rot=0, title="MdAPE (%) by price quintile")
bands.pivot(index="band", columns="model", values="median_pct_error").plot(kind="bar", ax=axes[1], rot=0, title="Median signed error (%) by quintile (> 0 = over-valued)")
axes[1].axhline(0, color="black", linewidth=0.8)
plt.tight_layout(); plt.show()
bands.pivot(index="group", columns="model", values=["MdAPE", "median_pct_error"])

In [ ]:
county_tables = []
for n, r in results.items():
    t = evaluation.metrics_by_group(r["test"]["ClosePrice"], r["pred"], r["test"]["CountyOrParish"], min_n=200)
    county_tables.append(t.assign(model=LABELS[n]))
pd.concat(county_tables).pivot(index="group", columns="model", values="MdAPE").sort_values("Random Forest")

## 6. What the Random Forest relies on

In [ ]:
rf = results["random_forest"]["model"]
imp = pd.Series(rf.named_steps["model"].feature_importances_, index=rf.named_steps["preprocess"].get_feature_names_out())
imp.sort_values(ascending=False).head(15).to_frame("importance (impurity decrease)")

## 7. Findings

| Test 2026-04 | R² | MdAPE | MAPE | MAE | Within 10% |
|---|---|---|---|---|---|
| Linear Regression | 0.822 | 11.2% | 15.4% | $209k | 45% |
| Decision Tree | 0.858 | 9.3% | 13.6% | $185k | 53% |
| **Random Forest** | **0.896** | **7.6%** | **11.2%** | **$154k** | **61%** |

- **Random Forest is the best model.** It cuts MdAPE by 3.6 points versus the baseline (95% bootstrap interval 3.4–3.9), so the gain is far larger than noise. The Decision Tree's 1.9-point gain (1.7–2.3) is also real.
- **Stable over time:** Random Forest MdAPE stays at 7.5–7.8% across all four test months; every model keeps its ranking in every month.
- **All three picked the 24-month window** on validation; more history helps every model.
- **Price bands:** Random Forest is best in every quintile and roughly halves the baseline's bias on cheap homes (+2.1% vs +4.9%). Luxury homes are still the weak spot (MdAPE 11.3%, under-valued by ~7%).
- **Geography:** Random Forest is best in every county; the Inland Empire is easiest (Riverside 6.0%, San Bernardino 6.1%), Alameda hardest (10.5%).
- **What it relies on:** location dominates (ZIP and city encodings ≈ 57% of importance), then living area (11%). Impurity importance favors high-cardinality inputs, so treat these as rough.

**Strengths and weaknesses**

| Model | Strengths | Weaknesses |
|---|---|---|
| Linear Regression | Fast, interpretable coefficients, small file | Can't bend at price extremes: over-values cheap homes, under-values luxury |
| Decision Tree | Captures non-linear effects and interactions; still fast | One tree is unstable; leaf averages give step-like predictions |
| Random Forest | Best on every metric, band and county; stable over time | 1.1 GB saved model (leaf size 1), ~15 min to run the full selection; harder to explain |

**Next:** Week 6 feature engineering (bed/bath ratio, flooring, school-district spatial join), then gradient boosting in Week 7. For the Week 9 app, a larger minimum leaf size or gradient boosting would give a much smaller model file.